In [1]:
import polars as pl
import requests
import xml.etree.ElementTree as ET
import numpy as np
import duckdb
import time
import plotly.graph_objects as go
from IPython.display import clear_output

In [2]:
import kagglehub
import os

path = kagglehub.dataset_download("sobhanmoosavi/us-accidents")
csv_file = os.path.join(path, "US_Accidents_March23.csv")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'us-accidents' dataset.
Path to dataset files: /kaggle/input/us-accidents


In [3]:

os.listdir(path)

['US_Accidents_March23.csv']

# **Preprocessing**

In [4]:
lazy_df = (
    pl.scan_csv(csv_file, try_parse_dates=False, infer_schema_length=10000)
    .with_columns(
        pl.col("Start_Time").str.to_datetime(strict=False)
    )
    .filter(
        pl.col("Start_Time").dt.year().is_between(2020, 2023)
    )
)

df = lazy_df.collect()
print(f"Rows after year filter: {df.height:,}")

Rows after year filter: 4,751,751


In [5]:
print("Schema")
print(df.schema)

print("Shape")
print(df.shape)

print("Null Counts")
print(df.null_count())

print("Summary Statistics")
print(df.describe())

print("Duplicates")
print("Duplicate rows:", df.is_duplicated().sum())

Schema
Schema({'ID': String, 'Source': String, 'Severity': Int64, 'Start_Time': Datetime(time_unit='us', time_zone=None), 'End_Time': String, 'Start_Lat': Float64, 'Start_Lng': Float64, 'End_Lat': String, 'End_Lng': String, 'Distance(mi)': Float64, 'Description': String, 'Street': String, 'City': String, 'County': String, 'State': String, 'Zipcode': String, 'Country': String, 'Timezone': String, 'Airport_Code': String, 'Weather_Timestamp': String, 'Temperature(F)': Float64, 'Wind_Chill(F)': Float64, 'Humidity(%)': Float64, 'Pressure(in)': Float64, 'Visibility(mi)': Float64, 'Wind_Direction': String, 'Wind_Speed(mph)': Float64, 'Precipitation(in)': Float64, 'Weather_Condition': String, 'Amenity': Boolean, 'Bump': Boolean, 'Crossing': Boolean, 'Give_Way': Boolean, 'Junction': Boolean, 'No_Exit': Boolean, 'Railway': Boolean, 'Roundabout': Boolean, 'Station': Boolean, 'Stop': Boolean, 'Traffic_Calming': Boolean, 'Traffic_Signal': Boolean, 'Turning_Loop': Boolean, 'Sunrise_Sunset': String, 

In [6]:
year_dist = (
    df
    .with_columns(pl.col("Start_Time").dt.year().alias("Year"))
    .group_by("Year")
    .len()
    .sort("Year")
)
print("Accidents per Year")
print(year_dist)

Accidents per Year
shape: (4, 2)
┌──────┬─────────┐
│ Year ┆ len     │
│ ---  ┆ ---     │
│ i32  ┆ u32     │
╞══════╪═════════╡
│ 2020 ┆ 1178913 │
│ 2021 ┆ 1563753 │
│ 2022 ┆ 1762452 │
│ 2023 ┆ 246633  │
└──────┴─────────┘


In [7]:
for col, null_count in zip(df.columns, df.null_count().row(0)):
    print(f"{col}: {null_count:,}")

ID: 0
Source: 0
Severity: 0
Start_Time: 0
End_Time: 0
Start_Lat: 0
Start_Lng: 0
End_Lat: 1,156,511
End_Lng: 1,156,511
Distance(mi): 0
Description: 4
Street: 10,863
City: 170
County: 0
State: 0
Zipcode: 1,100
Country: 0
Timezone: 4,710
Airport_Code: 17,076
Weather_Timestamp: 83,638
Temperature(F): 107,878
Wind_Chill(F): 145,116
Humidity(%): 115,058
Pressure(in): 92,626
Visibility(mi): 111,487
Wind_Direction: 130,210
Wind_Speed(mph): 130,166
Precipitation(in): 203,689
Weather_Condition: 107,604
Amenity: 0
Bump: 0
Crossing: 0
Give_Way: 0
Junction: 0
No_Exit: 0
Railway: 0
Roundabout: 0
Station: 0
Stop: 0
Traffic_Calming: 0
Traffic_Signal: 0
Turning_Loop: 0
Sunrise_Sunset: 23,139
Civil_Twilight: 23,139
Nautical_Twilight: 23,139
Astronomical_Twilight: 23,139


In [8]:
cols_to_drop = [
    "End_Lat", "End_Lng",          # mostly null and string type
    "Description",                  # free text
    "Zipcode", "Country",           # always US'; Zipcode redundant with City/State
    "Turning_Loop",                 # all False — zero variance
    "Airport_Code",                 # we have city/state
    "Wind_Chill(F)",               # correlated with Temperature, adds redundancy
    "Civil_Twilight", "Nautical_Twilight", "Astronomical_Twilight",  # redundant with Sunrise_Sunset
]
cols_to_drop = [c for c in cols_to_drop if c in df.columns]
df = df.drop(cols_to_drop)
print(f"Columns after drop: {df.width}")

Columns after drop: 35


In [9]:
print(df.columns)


['ID', 'Source', 'Severity', 'Start_Time', 'End_Time', 'Start_Lat', 'Start_Lng', 'Distance(mi)', 'Street', 'City', 'County', 'State', 'Timezone', 'Weather_Timestamp', 'Temperature(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Direction', 'Wind_Speed(mph)', 'Precipitation(in)', 'Weather_Condition', 'Amenity', 'Bump', 'Crossing', 'Give_Way', 'Junction', 'No_Exit', 'Railway', 'Roundabout', 'Station', 'Stop', 'Traffic_Calming', 'Traffic_Signal', 'Sunrise_Sunset']


In [10]:
#drop city and sunrise_sunset nulls
before = df.height
df = df.drop_nulls(subset=["City", "Sunrise_Sunset"])
print(f"Rows dropped (City/Sunrise_Sunset nulls): {before - df.height:,}")

Rows dropped (City/Sunrise_Sunset nulls): 23,139


In [11]:
#fill nulls wit median
numeric_fills = {
    "Temperature(F)":   df["Temperature(F)"].median(),
    "Humidity(%)":      df["Humidity(%)"].median(),
    "Pressure(in)":     df["Pressure(in)"].median(),
    "Visibility(mi)":   df["Visibility(mi)"].median(),
    "Wind_Speed(mph)":  df["Wind_Speed(mph)"].median(),
}
df = df.with_columns([
    pl.col(col).fill_null(val) for col, val in numeric_fills.items()
])


In [12]:
df = df.with_columns(
    pl.col("Precipitation(in)").fill_null(0.0)
)


In [13]:
for col in ["Street", "Timezone", "Wind_Direction", "Weather_Condition"]:
    df = df.with_columns(pl.col(col).fill_null("Unknown"))


In [14]:
#delete impossible values
before = df.height
df = df.filter(
    (pl.col("Temperature(F)").is_between(-60, 140)) &
    (pl.col("Humidity(%)").is_between(0, 100)) &
    (pl.col("Visibility(mi)") >= 0) &
    (pl.col("Wind_Speed(mph)") >= 0) &
    (pl.col("Precipitation(in)") >= 0) &
    (pl.col("Distance(mi)") >= 0)
)
print(f"Rows dropped with invalid values: {before - df.height:,}")
print(f"Clean rows: {df.height:,}")

Rows dropped with invalid values: 26
Clean rows: 4,728,586


In [15]:
df = df.drop("Weather_Timestamp")

In [16]:
for col, null_count in zip(df.columns, df.null_count().row(0)):
    if null_count > 0:
        print(f"  {col}: {null_count:,}")
if df.null_count().sum_horizontal().sum() == 0: print("no nulls left.")

no nulls left.


# **EDA**

In [17]:
import plotly.express as px
import plotly.graph_objects as go

In [17]:
df = df.with_columns([
    pl.col("Start_Time").dt.hour().alias("Hour"),
    pl.col("Start_Time").dt.month().alias("Month"),
    pl.col("Start_Time").dt.year().alias("Year"),
])

In [19]:
print(df["Severity"].dtype)

Int64


In [20]:
severity_dist = (
    df.group_by("Severity")
    .len()
    .sort("Severity")
    .to_pandas()
)

fig1 = px.bar(
    severity_dist,
    x="Severity",
    y="len",
    title="Accident Count by Severity Level",
    labels={"len": "Number of Accidents", "Severity": "Severity Level"},
    color="Severity",
    color_continuous_scale="Reds",
    text="len"
)
fig1.update_traces(texttemplate="%{text:,}", textposition="outside")
fig1.update_layout(coloraxis_showscale=False)
fig1.show()

In [21]:
hourly = (
    df.group_by("Hour")
    .len()
    .sort("Hour")
    .to_pandas()
)

fig2 = px.line(
    hourly,
    x="Hour",
    y="len",
    title="Accidents by Hour of Day",
    labels={"len": "Number of Accidents", "Hour": "Hour of Day"},
    markers=True
)
fig2.update_traces(line_color="#378ADD")
fig2.show()

In [22]:
import calendar
monthly = (
    df.group_by("Month")
    .len()
    .sort("Month")
    .to_pandas()
)
# Add month name column
monthly["Month_Name"] = monthly["Month"].apply(lambda x: calendar.month_abbr[x])

fig3 = px.bar(
    monthly,
    x="len",
    y="Month_Name",
    orientation="h",
    title="Accidents by Month (2020–2023)",
    labels={"len": "Number of Accidents", "Month_Name": "Month"},
    text="len",
    color_discrete_sequence=["#378ADD"],
    category_orders={"Month_Name": [calendar.month_abbr[i] for i in range(1, 13)]}
)
fig3.update_traces(texttemplate="%{text:,}", textposition="outside")
fig3.update_layout(
    yaxis=dict(autorange="reversed"),
    xaxis_title="Number of Accidents",
    yaxis_title="Month"
)
fig3.show()

In [23]:
day_night = (
    df.group_by("Sunrise_Sunset")
    .len()
    .to_pandas()
)

fig4 = px.pie(
    day_night,
    names="Sunrise_Sunset",
    values="len",
    title="Accidents: Day vs Night",
    color_discrete_sequence=["#FFC300", "#1F4E8C"]
)
fig4.show()


In [24]:
top_states = (
    df.group_by("State")
    .len()
    .sort("len", descending=True)
    .head(10)
    .to_pandas()
)

fig5 = px.bar(
    top_states,
    x="len",
    y="State",
    orientation="h",
    title="Top 10 States by Accident Count",
    labels={"len": "Number of Accidents", "State": "State"},
    color="len",
    color_continuous_scale="Oranges",
    text="len"
)
fig5.update_traces(texttemplate="%{text:,}", textposition="outside")
fig5.update_layout(
    coloraxis_showscale=False,
    yaxis=dict(autorange="reversed")
)
fig5.show()

In [25]:
top_10_weather = (
    df.group_by("Weather_Condition")
    .len()
    .sort("len", descending=True)
    .head(10)
    .get_column("Weather_Condition")
    .to_list()
)

weather_sev = (
    df.filter(pl.col("Weather_Condition").is_in(top_10_weather))
    .group_by(["Weather_Condition", "Severity"])
    .len()
    .to_pandas()
)

weather_sev["Severity"] = weather_sev["Severity"].astype(str)




fig6 = px.bar(
    weather_sev,
    x="Weather_Condition",
    y="len",
    color="Severity",
    title="Top 10 Weather Conditions vs Severity",
    labels={"len": "Number of Accidents", "Weather_Condition": "Weather Condition"},
    barmode="stack",
    color_discrete_map={
        "1": "#2ecc71",
        "2": "#f39c12",
        "3": "#e74c3c",
        "4": "#8e44ad"
    }
)
fig6.update_layout(xaxis_tickangle=-35)
fig6.show()

In [26]:
numeric_cols = [
    "Severity", "Temperature(F)", "Humidity(%)",
    "Pressure(in)", "Visibility(mi)",
    "Wind_Speed(mph)", "Precipitation(in)", "Distance(mi)"
]

corr_matrix = (
    df.select(numeric_cols)
    .to_pandas()
    .corr()
    .round(2)
)

fig7 = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns.tolist(),
    y=corr_matrix.columns.tolist(),
    colorscale="RdBu",
    zmid=0,
    text=corr_matrix.values,
    texttemplate="%{text}",
))
fig7.update_layout(
    title="Correlation Heatmap — Numeric Features vs Severity",
    xaxis_tickangle=-35,
    height=500
)
fig7.show()

# **Automation - PySpark - Classification**

In [21]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import plotly.graph_objects as go
import plotly.express as px
import os, tempfile, warnings
warnings.filterwarnings("ignore")

In [22]:
spark          = None
sdf            = None
sdf_clean      = None
pipeline_model = None
predictions    = None
accuracy       = None
precision      = None
recall         = None
f1             = None

In [18]:
def load():
    global spark, sdf
    spark = SparkSession.builder \
        .appName("USAccidents_NexusStream") \
        .master("local[*]") \
        .config("spark.driver.memory", "4g") \
        .getOrCreate()
    spark.sparkContext.setLogLevel("ERROR")

    tmp_path = os.path.join(tempfile.gettempdir(), "us_accidents_clean.parquet")
    df.write_parquet(tmp_path)
    sdf = spark.read.parquet(tmp_path)
    print(f"Loaded: {sdf.count():,} rows x {len(sdf.columns)} columns")

In [19]:
def feature_engineering():
    global sdf_clean

    #Feature Engineering 1
    sdf_fe = sdf \
        .withColumn("Hour",      F.hour("Start_Time")) \
        .withColumn("Month",     F.month("Start_Time")) \
        .withColumn("DayOfWeek", F.dayofweek("Start_Time")) \
        .withColumn("Year",      F.year("Start_Time")) \
        .withColumn(
            "Is_RushHour",
            F.when(
                (F.col("Hour").between(7, 9)) |
                (F.col("Hour").between(16, 19)), 1
            ).otherwise(0)
        ) \
        .withColumn(
            "Is_Weekend",
            F.when(F.col("DayOfWeek").isin([1, 7]), 1).otherwise(0)
        )

    print("Feature Engineering Step 1 — Time Features: Done")

    #Engineering Feature 2
    road_features = [
        "Amenity", "Bump", "Crossing", "Give_Way", "Junction",
        "No_Exit", "Railway", "Roundabout", "Station", "Stop",
        "Traffic_Calming", "Traffic_Signal"
    ]

    sdf_fe = sdf_fe \
        .withColumn(
            "Visibility_Category",
            F.when(F.col("Visibility(mi)") < 1, 0)
             .when(F.col("Visibility(mi)") < 5, 1)
             .otherwise(2)
        ) \
        .withColumn(
            "Temp_Category",
            F.when(F.col("Temperature(F)") < 32, 0)
             .when(F.col("Temperature(F)") < 77, 1)
             .otherwise(2)
        ) \
        .withColumn(
            "Road_Complexity_Score",
            sum(F.col(c).cast("int") for c in road_features)
        ) \
        .withColumn(
            "Is_HeavyRain",
            F.when(F.col("Precipitation(in)") > 0.3, 1).otherwise(0)
        )

    print("Feature Engineering Step 2 — Weather & Road Features: Done")

    # Drop replaced columns
    cols_to_drop = [
        "Start_Time", "End_Time", "Street", "County", "Timezone",
        "Wind_Direction", "Start_Lat", "Start_Lng",
        "ID", "Source", "City"
    ] + road_features
    cols_to_drop = [c for c in cols_to_drop if c in sdf_fe.columns]
    sdf_fe = sdf_fe.drop(*cols_to_drop)

    # Cast Severity and create 0-indexed label
    sdf_fe = sdf_fe \
        .withColumn("Severity", F.col("Severity").cast("int")) \
        .withColumn("label", F.col("Severity") - 1)

    # Drop nulls
    sdf_clean = sdf_fe.na.drop()
    print(f"Clean rows: {sdf_clean.count():,}")

In [31]:
def explore():
    print("── Schema ──")
    sdf_clean.printSchema()

    print("── Summary Statistics ──")
    sdf_clean.describe().show()

    print("── Severity Distribution ──")
    sdf_clean.groupBy("Severity") \
        .count() \
        .orderBy("Severity") \
        .show()

    print("── Null Counts ──")
    sdf_clean.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in sdf_clean.columns
    ]).show()

In [32]:
def train_evaluate():
    global pipeline_model, predictions, accuracy, precision, recall, f1

    # Categorical columns
    cat_cols   = ["State", "Sunrise_Sunset", "Weather_Condition"]
    cat_cols   = [c for c in cat_cols if c in sdf_clean.columns]
    index_cols = [c + "_idx" for c in cat_cols]

    # Numeric feature columns
    num_cols = [
        "Temperature(F)", "Humidity(%)", "Pressure(in)",
        "Visibility(mi)", "Wind_Speed(mph)", "Precipitation(in)",
        "Distance(mi)", "Hour", "Month", "DayOfWeek", "Year",
        "Is_RushHour", "Is_Weekend", "Visibility_Category",
        "Temp_Category", "Road_Complexity_Score", "Is_HeavyRain"
    ]
    all_feature_cols = num_cols + index_cols

    # Train/test split
    train_df, test_df = sdf_clean.randomSplit([0.8, 0.2], seed=42)
    print(f"Train rows: {train_df.count():,}")
    print(f"Test rows : {test_df.count():,}")

    # Pipeline stages
    indexers = [
        StringIndexer(inputCol=c, outputCol=i, handleInvalid="keep")
        for c, i in zip(cat_cols, index_cols)
    ]

    assembler = VectorAssembler(
        inputCols=all_feature_cols,
        outputCol="raw_features",
        handleInvalid="skip"
    )

    scaler = StandardScaler(
        inputCol="raw_features",
        outputCol="features",
        withStd=True,
        withMean=False
    )

    rf = RandomForestClassifier(
        featuresCol="features",
        labelCol="label",
        numTrees=100,
        maxDepth=10,
        seed=42
    )

    # Build and fit pipeline
    pipeline = Pipeline(stages=indexers + [assembler, scaler, rf])
    print("Training Random Forest pipeline...")
    pipeline_model = pipeline.fit(train_df)
    print("Training complete.")

    # Predictions
    predictions = pipeline_model.transform(test_df)

    # Evaluate
    def evaluate(metric):
        return MulticlassClassificationEvaluator(
            labelCol="label",
            predictionCol="prediction",
            metricName=metric
        ).evaluate(predictions)

    accuracy  = evaluate("accuracy")
    precision = evaluate("weightedPrecision")
    recall    = evaluate("weightedRecall")
    f1        = evaluate("f1")

    # Metrics table
    go.Figure(go.Table(
        header=dict(
            values=["Metric", "Value"],
            fill_color="#1F4E8C",
            font=dict(color="white", size=13),
            align="left"
        ),
        cells=dict(
            values=[
                ["Accuracy", "Precision", "Recall", "F1 Score"],
                [f"{accuracy:.4f}", f"{precision:.4f}",
                 f"{recall:.4f}", f"{f1:.4f}"]
            ],
            fill_color=[["#EAF3FB", "white"] * 2],
            align="left",
            font=dict(size=13),
            height=30
        )
    )).update_layout(
        title="Model Evaluation — Random Forest Classifier",
        height=240
    ).show()

    # Feature importances
    rf_model    = pipeline_model.stages[-1]
    importances = rf_model.featureImportances.toArray()
    feat_imp    = sorted(
        zip(all_feature_cols, importances),
        key=lambda x: x[1],
        reverse=True
    )[:10]

    px.bar(
        x=[f[1] for f in feat_imp],
        y=[f[0] for f in feat_imp],
        orientation="h",
        title="Top 10 Feature Importances — Random Forest",
        labels={"x": "Importance Score", "y": "Feature"},
        color_discrete_sequence=["#1F4E8C"]
    ).update_layout(yaxis=dict(autorange="reversed")).show()

    # Save predictions
    predictions_pd = predictions.select(
        "Severity", "label", "prediction",
        "Hour", "Month", "State",
        "Temperature(F)", "Visibility(mi)",
        "Road_Complexity_Score", "Is_RushHour",
        "Is_Weekend", "Is_HeavyRain"
    ).sample(fraction=0.3, seed=42).toPandas()
    predictions_pd.to_csv("/content/predictions.csv", index=False)

    # Save feature engineered dataset
    sdf_clean.select(
        "Severity", "label",
        "Hour", "Month", "DayOfWeek", "Year",
        "Is_RushHour", "Is_Weekend",
        "Visibility_Category", "Temp_Category",
        "Road_Complexity_Score", "Is_HeavyRain",
        "Temperature(F)", "Humidity(%)",
        "Pressure(in)", "Visibility(mi)",
        "Wind_Speed(mph)", "Precipitation(in)",
        "Distance(mi)", "State",
        "Sunrise_Sunset", "Weather_Condition"
    ).toPandas().to_csv("/content/us_accidents_featured.csv", index=False)


In [33]:
def pipeline():
    load()
    feature_engineering()
    explore()
    train_evaluate()

In [ ]:
pipeline()

Loaded: 4,728,586 rows x 37 columns
Feature Engineering Step 1 — Time Features: Done
Feature Engineering Step 2 — Weather & Road Features: Done
Clean rows: 4,728,586
── Schema ──
root
 |-- Severity: integer (nullable = true)
 |-- Distance(mi): double (nullable = true)
 |-- State: string (nullable = true)
 |-- Temperature(F): double (nullable = true)
 |-- Humidity(%): double (nullable = true)
 |-- Pressure(in): double (nullable = true)
 |-- Visibility(mi): double (nullable = true)
 |-- Wind_Speed(mph): double (nullable = true)
 |-- Precipitation(in): double (nullable = true)
 |-- Weather_Condition: string (nullable = true)
 |-- Sunrise_Sunset: string (nullable = true)
 |-- Hour: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- Is_RushHour: integer (nullable = false)
 |-- Is_Weekend: integer (nullable = false)
 |-- Visibility_Category: integer (nullable = false)
 |-- Temp_Category: integer 

In [ ]:
# Save cleaned dataset
df.write_csv("/content/us_accidents_clean.csv")

In [23]:
load()
feature_engineering()

# Save immediately
sdf_clean.sample(fraction=0.3, seed=42) \
         .toPandas() \
         .to_csv("/content/us_accidents_featured.csv", index=False)
print("✓ Saved")

Loaded: 4,728,586 rows x 37 columns
Feature Engineering Step 1 — Time Features: Done
Feature Engineering Step 2 — Weather & Road Features: Done
Clean rows: 4,728,586
✓ Saved
